# OpenAgent Eval: Corpus Health Auditor & Related Modules

---

## What This Notebook Covers

This notebook provides a **hands-on guide** to the three new v0.3.0 modules:

| Module | Purpose | Key Classes |
|--------|---------|-------------|
| **Corpus** | Scan knowledge bases for contradictions, staleness, duplicates, and coverage gaps | `CorpusAuditor`, `ContradictionDetector`, `StalenessDetector`, `DuplicateDetector`, `CoverageAnalyzer` |
| **Diagnosis** | Attribute blame when RAG evaluations fail | `DiagnosisAnalyzer`, `BlameAttribution`, `ChunkingQualityAnalyzer` |
| **Synthesis** | Auto-generate Q&A test cases from documents | `SyntheticDataGenerator`, `QuestionGenerator`, `AdversarialTestCaseGenerator` |

---

### Prerequisites

- **Python 3.11+**
- **Install OpenAgent Eval**:

```bash
pip install openagent-eval
```

---

# 1. Installation

Install OpenAgent Eval with the `corpus` extras for full analyzer support (numpy, scikit-learn, sentence-transformers).

In [3]:
import subprocess
import sys

# Install with corpus + providers extras (includes groq, sentence-transformers)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "openagent-eval[corpus,providers]"],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'openagent-eval[corpus,providers]'], returncode=0)

---

# 2. Corpus Module — Overview

The **Corpus Health Auditor** scans your knowledge base *before* connecting to a RAG pipeline. It detects issues that no existing evaluation tool can see:

```
┌─────────────────────────────────────────────────────────────┐
│                   CorpusAuditor (Orchestrator)              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌──────────────────┐  ┌──────────────────┐                │
│  │ Contradiction    │  │ Staleness        │                │
│  │ Detector         │  │ Detector         │                │
│  │ (LLM-as-Judge)   │  │ (Timestamp)      │                │
│  └──────────────────┘  └──────────────────┘                │
│                                                             │
│  ┌──────────────────┐  ┌──────────────────┐                │
│  │ Duplicate        │  │ Coverage         │                │
│  │ Detector         │  │ Analyzer         │                │
│  │ (Embeddings)     │  │ (Clustering)     │                │
│  └──────────────────┘  └──────────────────┘                │
│                                                             │
│  Output: AuditReport (health_score, issues, summary)       │
└─────────────────────────────────────────────────────────────┘
```

### Key Concepts

| Concept | Description |
|---------|-------------|
| `CorpusDocument` | A single document with `doc_id`, `content`, and `metadata` |
| `CorpusIssue` | A detected issue with `issue_type`, `severity`, `title`, `description` |
| `AuditReport` | Complete report with `health_score` (0.0-1.0), issues, and summary |
| `IssueType` | Enum: `CONTRADICTION`, `STALENESS`, `DUPLICATE`, `COVERAGE_GAP` |
| `IssueSeverity` | Enum: `LOW`, `MEDIUM`, `HIGH`, `CRITICAL` |

---

# 3. Corpus Module — Prepare Sample Documents

We'll create a small corpus with **deliberate issues** to demonstrate each analyzer.

In [4]:
from openagent_eval.corpus.models import CorpusDocument

# Sample corpus with intentional issues for demonstration
documents = [
    # --- Contradiction pair: these two documents disagree on Python's type system ---
    CorpusDocument(
        doc_id="python_intro",
        content=(
            "Python is a dynamically typed language. Variables do not need "
            "explicit type declarations, and types are checked at runtime. "
            "This flexibility makes Python easy to learn and rapid to "
            "prototype with."
        ),
        metadata={"topic": "python", "source": "intro.md"},
    ),
    CorpusDocument(
        doc_id="python_static",
        content=(
            "Python is a statically typed language. All variables must be "
            "annotated with type hints, and types are verified at compile "
            "time before execution. This makes Python suitable for large "
            "enterprise codebases."
        ),
        metadata={"topic": "python", "source": "advanced.md"},
    ),

    # --- Stale document (old timestamp) ---
    CorpusDocument(
        doc_id="rag_2021",
        content=(
            "Retrieval-Augmented Generation (RAG) was introduced in 2020. "
            "Early implementations used BERT-based retrievers and GPT-2 "
            "generators. The approach was limited by small context windows."
        ),
        metadata={
            "topic": "rag",
            "updated_at": "2021-03-15",
            "source": "rag_history.md",
        },
    ),
    CorpusDocument(
        doc_id="rag_current",
        content=(
            "Modern RAG systems use dense retrievers with transformer "
            "embeddings and large context LLMs. They support real-time "
            "knowledge grounding with sub-second latency."
        ),
        metadata={"topic": "rag", "updated_at": "2026-01-10", "source": "rag_modern.md"},
    ),

    # --- Near-duplicate pair ---
    CorpusDocument(
        doc_id="embeddings_v1",
        content=(
            "Vector embeddings are dense numerical representations of text "
            "that capture semantic meaning. In semantic search, both queries "
            "and documents are converted to embeddings, and similar vectors "
            "indicate semantically related content."
        ),
        metadata={"topic": "embeddings", "source": "embeddings.md"},
    ),
    CorpusDocument(
        doc_id="embeddings_v2",
        content=(
            "Vector embeddings are dense numerical representations of text "
            "that capture semantic meaning. In semantic search, both queries "
            "and documents are converted to embeddings, and similar vectors "
            "indicate semantically related content. Updated 2025."
        ),
        metadata={"topic": "embeddings", "source": "embeddings_updated.md"},
    ),

    # --- Isolated topic (potential coverage gap) ---
    CorpusDocument(
        doc_id="quantum_intro",
        content=(
            "Quantum computing uses qubits that can exist in superposition, "
            "enabling parallel computation. Quantum algorithms like Shor's "
            "and Grover's offer exponential speedups for specific problems."
        ),
        metadata={"topic": "quantum", "source": "quantum.md"},
    ),

    # --- Healthy documents ---
    CorpusDocument(
        doc_id="vectordb",
        content=(
            "A vector database stores high-dimensional embeddings and supports "
            "similarity search via cosine or dot-product distance. Popular "
            "options include Chroma, Qdrant, Pinecone, and FAISS."
        ),
        metadata={"topic": "vectordb", "source": "vectordb.md"},
    ),
    CorpusDocument(
        doc_id="chunking",
        content=(
            "Chunking strategies determine how documents are split into "
            "smaller pieces for embedding. Common approaches include fixed-size, "
            "sentence-based, and semantic chunking."
        ),
        metadata={"topic": "chunking", "source": "chunking.md"},
    ),
]

print(f"Prepared {len(documents)} sample documents:")
for doc in documents:
    print(f"  - {doc.doc_id}: {doc.metadata.get('topic')} ({len(doc.content)} chars)")

Prepared 9 sample documents:
  - python_intro: python (196 chars)
  - python_static: python (203 chars)
  - rag_2021: rag (182 chars)
  - rag_current: rag (159 chars)
  - embeddings_v1: embeddings (227 chars)
  - embeddings_v2: embeddings (241 chars)
  - quantum_intro: quantum (187 chars)
  - vectordb: vectordb (180 chars)
  - chunking: chunking (165 chars)


---

# 4. Corpus Module — Run Individual Analyzers

Each analyzer can be used independently. Let's walk through all four.

## 4a. Staleness Detector

Checks document metadata for timestamps and flags documents older than a configurable threshold (default: 365 days).

In [5]:
import asyncio
from openagent_eval.corpus.staleness import StalenessDetector

# Flag documents older than 365 days
detector = StalenessDetector(staleness_days=365)
report = await detector.analyze(documents)

print(f"Health score: {report.health_score:.2f}")
print(f"Issues found: {len(report.issues)}")
print(f"Summary: {report.summary}")
print()
for issue in report.issues:
    print(f"  [{issue.severity.value}] {issue.title}")
    print(f"    {issue.description}")
    print()

Health score: 0.89
Issues found: 1
Summary: Found 1 stale document(s) across 9 documents. 1 critical (very old).

  [critical] Document is 1944 days old
    Document 'rag_2021' was last updated on 2021-03-15, which is 1944 days ago (threshold: 365 days).



## 4b. Duplicate Detector

Uses sentence-transformers embeddings + cosine similarity to find near-duplicate documents.

In [ ]:
from openagent_eval.corpus.duplicates import DuplicateDetector

# Find documents with >92% similarity
detector = DuplicateDetector(
    similarity_threshold=0.92,
    embedding_model="all-MiniLM-L6-v2",
)
report = await detector.analyze(documents)

print(f"Health score: {report.health_score:.2f}")
print(f"Issues found: {len(report.issues)}")
print(f"Summary: {report.summary}")
print()
for issue in report.issues:
    print(f"  [{issue.severity.value}] {issue.title}")
    print(f"    {issue.description}")
    print(f"    Similarity: {issue.metadata.get('similarity', 'N/A')}")
    print()

## 4c. Coverage Analyzer

Clusters documents to detect thematic coverage gaps — topics with too few documents.

In [ ]:
from openagent_eval.corpus.coverage import CoverageAnalyzer

# Flag clusters with fewer than 2 documents
analyzer = CoverageAnalyzer(
    min_cluster_size=2,
    embedding_model="all-MiniLM-L6-v2",
)
report = await analyzer.analyze(documents)

print(f"Health score: {report.health_score:.2f}")
print(f"Issues found: {len(report.issues)}")
print(f"Summary: {report.summary}")
print()

# Show cluster breakdown
clusters = report.metadata.get("clusters", [])
print(f"Topics discovered: {report.metadata.get('num_topics', 0)}")
for cluster in clusters:
    print(f"  Cluster {cluster['cluster_id']}: '{cluster['topic']}' ({cluster['size']} docs)")
    print(f"    Docs: {cluster['document_ids']}")
print()
for issue in report.issues:
    print(f"  [{issue.severity.value}] {issue.title}")
    print(f"    {issue.description}")
    print()

## 4d. Contradiction Detector

Uses LLM-as-Judge to compare document pairs and detect contradictory information. Requires an LLM provider.

> **Note**: Without an LLM provider, the detector returns a report indicating that an LLM is required. We'll demonstrate with a mock provider below.

In [ ]:
import os
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()

from openagent_eval.corpus.contradiction import ContradictionDetector
from openagent_eval.providers.llm.groq import Groq

# Create a real Groq LLM provider using the API key from .env
groq_api_key = os.environ.get("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file. Please add it to .env")

real_llm = Groq(
    api_key=groq_api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.0,
    max_tokens=1024,
)
print(f"Groq LLM initialized: model={real_llm.model}")

detector = ContradictionDetector(
    llm_provider=real_llm,
    max_pairs=10,
)
report = await detector.analyze(documents)

print(f"Health score: {report.health_score:.2f}")
print(f"Issues found: {len(report.issues)}")
print(f"Summary: {report.summary}")
print(f"Pairs compared: {report.metadata.get('pairs_compared', 'N/A')}")
print()
for issue in report.issues:
    print(f"  [{issue.severity.value}] {issue.title}")
    print(f"    {issue.description}")
    print(f"    Confidence: {issue.metadata.get('confidence', 'N/A')}")
    print()

---

# 5. Corpus Module — Full Audit with CorpusAuditor

The `CorpusAuditor` orchestrates all analyzers and merges their results into a single `AuditReport`.

### Option A: Audit a directory of files

In [ ]:
from openagent_eval.corpus import CorpusAuditor

# Audit the sample corpus directory
auditor = CorpusAuditor(
    llm_provider=real_llm,
    checks=["staleness", "duplicate", "coverage"],  # Skip contradiction for speed
    staleness_days=365,
    similarity_threshold=0.92,
)

report = await auditor.audit("data/")

print("=" * 60)
print("CORPUS AUDIT REPORT")
print("=" * 60)
print(f"Health Score:  {report.health_score:.2f} / 1.00")
print(f"Documents:     {report.total_documents}")
print(f"Issues Found:  {len(report.issues)}")
print(f"Checks Run:    {', '.join(report.checks_performed)}")
print(f"Healthy:       {report.healthy}")
print()
print(f"Summary: {report.summary}")
print()

# Group issues by type
by_type = report.issues_by_type
for issue_type, issues in by_type.items():
    print(f"\n--- {issue_type.value.upper()} ({len(issues)} issues) ---")
    for issue in issues:
        print(f"  [{issue.severity.value}] {issue.title}")
        print(f"    {issue.description[:120]}...")

### Option B: Audit from Python objects directly

If your documents are already in memory as `CorpusDocument` objects, you can run individual analyzers directly (as shown in Section 4) and merge reports manually.

In [ ]:
# Run multiple analyzers and merge results manually
from openagent_eval.corpus.staleness import StalenessDetector
from openagent_eval.corpus.duplicates import DuplicateDetector
from openagent_eval.corpus.models import AuditReport, CorpusIssue, IssueSeverity

staleness_report = await StalenessDetector(staleness_days=365).analyze(documents)
duplicate_report = await DuplicateDetector(similarity_threshold=0.92).analyze(documents)

# Merge manually
all_issues = staleness_report.issues + duplicate_report.issues
avg_score = (staleness_report.health_score + duplicate_report.health_score) / 2

merged = AuditReport(
    corpus_path="in-memory",
    total_documents=len(documents),
    issues=all_issues,
    health_score=round(avg_score, 3),
    checks_performed=["staleness", "duplicate"],
    summary=f"Merged report: {len(all_issues)} total issues across 2 checks.",
)

print(f"Merged health score: {merged.health_score:.2f}")
print(f"Total issues: {len(merged.issues)}")
print(f"By severity: { {s.value: len(issues) for s, issues in merged.issues_by_severity.items()} }")

---

# 6. Corpus Module — Custom Analyzer

You can create your own analyzer by extending `BaseCorpusAnalyzer`.

In [ ]:
from openagent_eval.corpus.base import BaseCorpusAnalyzer
from openagent_eval.corpus.models import AuditReport, CorpusDocument, CorpusIssue, IssueSeverity, IssueType


class ShortDocumentDetector(BaseCorpusAnalyzer):
    """Custom analyzer that flags documents shorter than a threshold."""

    name = "short_document"
    description = "Detects documents that are suspiciously short"

    def __init__(self, min_length: int = 50) -> None:
        self.min_length = min_length

    async def analyze(self, documents: list[CorpusDocument], **kwargs) -> AuditReport:
        self.validate_inputs(documents)

        issues = []
        for doc in documents:
            if len(doc.content.strip()) < self.min_length:
                issues.append(
                    CorpusIssue(
                        issue_type=IssueType.COVERAGE_GAP,
                        severity=IssueSeverity.MEDIUM,
                        title=f"Short document: {doc.doc_id}",
                        description=(
                            f"Document '{doc.doc_id}' is only {len(doc.content)} chars. "
                            f"Minimum recommended: {self.min_length} chars."
                        ),
                        document_ids=[doc.doc_id],
                        metadata={"char_count": len(doc.content), "min_length": self.min_length},
                    )
                )

        health_score = max(1.0 - (len(issues) / max(len(documents), 1)), 0.0)

        return AuditReport(
            corpus_path="",
            total_documents=len(documents),
            issues=issues,
            health_score=round(health_score, 3),
            checks_performed=[self.name],
            summary=f"Checked {len(documents)} documents for minimum length ({self.min_length} chars). Found {len(issues)} short documents.",
        )


# Use the custom analyzer
custom = ShortDocumentDetector(min_length=100)
report = await custom.analyze(documents)

print(f"Health score: {report.health_score:.2f}")
print(f"Summary: {report.summary}")
for issue in report.issues:
    print(f"  [{issue.severity.value}] {issue.title} — {issue.description}")

---

# 7. Diagnosis Module — Blame Attribution

When your RAG evaluation fails, the **Diagnosis** module tells you *where* it failed: retrieval, generation, or chunking.

```
┌─────────────────────────────────────────────────────────────┐
│                  DiagnosisAnalyzer                          │
├─────────────────────────────────────────────────────────────┤
│  Input: Evaluation results (question, answer, metrics)     │
│                                                             │
│  ┌──────────────────┐  ┌──────────────────┐                │
│  │ BlameAttribution │  │ ChunkingQuality  │                │
│  │ (8 failure modes)│  │ Analyzer         │                │
│  └──────────────────┘  └──────────────────┘                │
│                                                             │
│  Output: DiagnosisReport                                   │
│    - blame_summary (retrieval/generation/chunking counts)   │
│    - failure_summary (by FailureMode)                       │
│    - recommendations (actionable next steps)                │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
from openagent_eval.diagnosis import DiagnosisAnalyzer

# Simulate evaluation results with failures
eval_results = [
    {
        "question": "What is RAG?",
        "answer": "RAG is a technique for building chatbots.",
        "contexts": ["RAG combines retrieval with generation to ground answers in documents."],
        "metrics": {
            "context_precision": 0.4,
            "context_recall": 0.3,
            "faithfulness": 0.9,
            "answer_relevancy": 0.5,
        },
        "metadata": {"latency_ms": 150.0},
    },
    {
        "question": "How do embeddings work?",
        "answer": "Embeddings are vectors that represent text meaning.",
        "contexts": [],  # No contexts retrieved!
        "metrics": {
            "context_precision": 0.0,
            "context_recall": 0.0,
            "faithfulness": 0.85,
            "answer_relevancy": 0.9,
        },
        "metadata": {"latency_ms": 200.0},
    },
    {
        "question": "What is chunking?",
        "answer": "The sky is blue and birds fly.",
        "contexts": [
            "Chunking splits documents into smaller pieces for embedding.",
            "Common strategies include fixed-size and semantic chunking.",
        ],
        "metrics": {
            "context_precision": 0.95,
            "context_recall": 0.9,
            "faithfulness": 0.1,  # LLM hallucinated!
            "answer_relevancy": 0.05,
        },
        "metadata": {"latency_ms": 120.0},
    },
    {
        "question": "What are vector databases?",
        "answer": "Vector databases store embeddings for similarity search.",
        "contexts": ["Vector databases store high-dimensional embeddings."],
        "metrics": {
            "context_precision": 0.85,
            "context_recall": 0.8,
            "faithfulness": 0.92,
            "answer_relevancy": 0.88,
        },
        "metadata": {"latency_ms": 100.0},
    },
]

# Run diagnosis
diagnosis = DiagnosisAnalyzer(blame_threshold=0.3)
report = diagnosis.analyze(eval_results)

print("=" * 60)
print("DIAGNOSIS REPORT")
print("=" * 60)
print(f"Items analyzed:  {report.total_items}")
print(f"Overall health:  {report.overall_health:.2f}")
print()
print("Blame Summary:")
for target, count in report.blame_summary.items():
    print(f"  {target}: {count} failures")
print()
print("Failure Summary:")
for mode, count in report.failure_summary.items():
    print(f"  {mode}: {count} occurrences")
print()
print("Failures Detected:")
for f in report.failures:
    print(f"  [{f.blame.value}] {f.mode.value} (conf: {f.confidence:.2f})")
    print(f"    Q: {f.question}")
    print(f"    Reason: {f.reason}")
    print()
print("Recommendations:")
for rec in report.recommendations:
    print(f"  - {rec}")

---

# 8. Synthesis Module — Generate Test Cases

The **Synthesis** module auto-generates Q&A test cases from your corpus, solving the "not enough test data" problem.

```
┌─────────────────────────────────────────────────────────────┐
│              SyntheticDataGenerator                         │
├─────────────────────────────────────────────────────────────┤
│  Input: Corpus path or raw text                             │
│                                                             │
│  ┌──────────────────┐  ┌──────────────────┐                │
│  │ Question         │  │ Adversarial      │                │
│  │ Generator        │  │ Test Generator   │                │
│  │ (standard Q&A)   │  │ (6 attack types) │                │
│  └──────────────────┘  └──────────────────┘                │
│                                                             │
│  Output: SyntheticDataset (test_cases, type_counts)         │
└─────────────────────────────────────────────────────────────┘
```

### Test Case Types

| Type | Description |
|------|-------------|
| `STANDARD` | Normal question-answer pair from document content |
| `UNANSWERABLE` | Question that *cannot* be answered from the context |
| `AMBIGUOUS` | Question with multiple valid interpretations |
| `MISLEADING` | Question that tempts the model to hallucinate |
| `MULTI_HOP` | Question requiring reasoning across multiple documents |
| `COUNTERFACTUAL` | Question based on a false premise |

## 8a. QuestionGenerator — Standard Q&A

In [ ]:
from openagent_eval.synthesis import QuestionGenerator

# Create a question generator with real Groq LLM
q_gen = QuestionGenerator(llm_provider=real_llm)

# Generate questions from a document chunk
context = (
    "Retrieval-Augmented Generation (RAG) combines a retrieval step with a "
    "generative model. The retriever searches a knowledge base for relevant "
    "documents, then the generator produces answers grounded in those documents."
)

test_cases = await q_gen.generate(
    context=context,
    count=3,
    source_document="rag_intro.md",
    chunk_index=0,
)

print(f"Generated {len(test_cases)} test cases:")
print()
for i, tc in enumerate(test_cases, 1):
    print(f"{i}. [{tc.test_type.value}] Q: {tc.question}")
    print(f"   A: {tc.ground_truth}")
    print(f"   Source: {tc.source_document}, Chunk: {tc.chunk_index}")
    print()

## 8b. AdversarialTestCaseGenerator — Attack Scenarios

In [ ]:
from openagent_eval.synthesis import AdversarialTestCaseGenerator
from openagent_eval.synthesis.models import TestCaseType

# Create adversarial generator with real Groq LLM
adv_gen = AdversarialTestCaseGenerator(llm_provider=real_llm)

# Generate specific adversarial types
context = (
    "Vector databases store high-dimensional embeddings and support "
    "similarity search via cosine or dot-product distance."
)

# Generate one of each adversarial type
for adv_type in [TestCaseType.UNANSWERABLE, TestCaseType.MISLEADING, TestCaseType.COUNTERFACTUAL]:
    cases = await adv_gen.generate(
        context=context,
        test_type=adv_type,
        count=1,
        source_document="vectordb.md",
    )
    if cases:
        tc = cases[0]
        print(f"[{tc.test_type.value}] Q: {tc.question}")
        print(f"  A: {tc.ground_truth}")
        print()

## 8c. SyntheticDataGenerator — Full Pipeline

In [ ]:
from openagent_eval.synthesis import SyntheticDataGenerator

# Create the full pipeline generator with real Groq LLM
generator = SyntheticDataGenerator(
    llm_provider=real_llm,
    chunk_size=2000,
    chunk_overlap=200,
    max_concurrent=3,
)

# Generate from raw text
text = (
    "Python is a high-level programming language known for its readability. "
    "It supports multiple paradigms including object-oriented, functional, "
    "and procedural programming. Python's standard library includes modules "
    "for file I/O, networking, and data processing."
)

dataset = await generator.generate_from_text(
    text=text,
    count=5,
    adversarial=True,
    adversarial_count_per_type=1,
    source_name="python_intro.md",
)

print(f"Total test cases: {dataset.total_count}")
print(f"Type breakdown:   {dataset.type_counts}")
print(f"Metadata:         {dataset.metadata}")
print()
for i, tc in enumerate(dataset.test_cases, 1):
    print(f"{i}. [{tc.test_type.value}] Q: {tc.question[:80]}")
    print(f"   A: {tc.ground_truth[:80]}")
    print()

---

# 9. End-to-End Workflow

Let's combine all three modules into a single workflow:

```
1. Audit corpus   →  Fix issues
2. Generate test data  →  Build evaluation dataset
3. Run evaluation  →  Get results
4. Diagnose failures  →  Fix pipeline
```

### Step 1: Audit the corpus

In [ ]:
from openagent_eval.corpus import CorpusAuditor
from openagent_eval.diagnosis import DiagnosisAnalyzer
from openagent_eval.synthesis import SyntheticDataGenerator
from openagent_eval.corpus.models import CorpusDocument

# --- Step 1: Audit the corpus ---
print("STEP 1: Corpus Audit")
print("-" * 40)

corpus_auditor = CorpusAuditor(
    llm_provider=real_llm,
    staleness_days=365,
    similarity_threshold=0.92,
    checks=["staleness", "duplicate", "coverage"],
)

corpus_report = await corpus_auditor.audit("data/")
print(f"  Health: {corpus_report.health_score:.2f}")
print(f"  Issues: {len(corpus_report.issues)}")
print(f"  Healthy: {corpus_report.healthy}")
print()

In [ ]:
# --- Step 2: Generate test cases ---
print("STEP 2: Generate Synthetic Test Cases")
print("-" * 40)

synth_gen = SyntheticDataGenerator(llm_provider=real_llm)

dataset = await synth_gen.generate(
    corpus_path="data/",
    count=10,
    adversarial=True,
    adversarial_count_per_chunk=1,
)

print(f"  Generated: {dataset.total_count} test cases")
print(f"  Types: {dataset.type_counts}")
print()

In [ ]:
# --- Step 3: Simulate evaluation results ---
print("STEP 3: Simulated Evaluation")
print("-" * 40)

# In a real scenario, you'd run: engine = Engine(config); results = await engine.run(dataset)
# Here we simulate results for demonstration

simulated_results = []
for tc in dataset.test_cases[:5]:
    simulated_results.append({
        "question": tc.question,
        "answer": tc.ground_truth,
        "contexts": [tc.context],
        "metrics": {
            "context_precision": 0.7,
            "context_recall": 0.8,
            "faithfulness": 0.6,
            "answer_relevancy": 0.5,
        },
        "metadata": {"latency_ms": 100.0},
    })

print(f"  Evaluated {len(simulated_results)} items")
print()

In [ ]:
# --- Step 4: Diagnose failures ---
print("STEP 4: Diagnose Failures")
print("-" * 40)

diagnosis_analyzer = DiagnosisAnalyzer(blame_threshold=0.3)
diagnosis_report = diagnosis_analyzer.analyze(simulated_results)

print(f"  Health: {diagnosis_report.overall_health:.2f}")
print(f"  Blame: {diagnosis_report.blame_summary}")
print(f"  Failures: {len(diagnosis_report.failures)}")
print()
print("  Recommendations:")
for rec in diagnosis_report.recommendations:
    print(f"    - {rec}")

---

# 10. Summary

## What We Covered

| Module | What It Does | When to Use |
|--------|-------------|-------------|
| **Corpus** | Scans knowledge base for contradictions, staleness, duplicates, coverage gaps | Before connecting corpus to RAG |
| **Diagnosis** | Attributes blame to retrieval/generation/chunking when evaluation fails | After running evaluation, when scores are low |
| **Synthesis** | Generates Q&A test cases from documents | When you don't have enough labeled test data |

## Key Classes

```python
# Corpus
from openagent_eval.corpus import CorpusAuditor, CorpusDocument, AuditReport
from openagent_eval.corpus import StalenessDetector, DuplicateDetector
from openagent_eval.corpus import CoverageAnalyzer, ContradictionDetector
from openagent_eval.corpus import BaseCorpusAnalyzer  # for custom analyzers

# Diagnosis
from openagent_eval.diagnosis import DiagnosisAnalyzer
from openagent_eval.diagnosis import BlameAttribution, ChunkingQualityAnalyzer
from openagent_eval.diagnosis import DiagnosisReport, FailureMode, BlameTarget

# Synthesis
from openagent_eval.synthesis import SyntheticDataGenerator
from openagent_eval.synthesis import QuestionGenerator, AdversarialTestCaseGenerator
from openagent_eval.synthesis import SyntheticDataset, TestCase, TestCaseType
```

## Next Steps

- Run `oaeval audit <corpus_path>` from the CLI for a quick corpus check
- Run `oaeval diagnose` after a failed evaluation run
- Integrate `SyntheticDataGenerator` into your CI/CD pipeline for continuous test data generation
- Build custom analyzers by extending `BaseCorpusAnalyzer`

---

*Generated by OpenAgent Eval v0.3.0*